#### Run on Colab to get faster runtime

In [ ]:
!pip install seqeval --use-pep517
!pip install transformers
!pip install datasets
!pip install accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=5cdcbe9d67b2249085e452ec3a9eee4d1088dfead9e491bd33e0546abcfa355e
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [ ]:
import json
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

In [ ]:
LABEL_MAP = "label_map.json"
TRAIN = "train.jsonl"
VAL = "val.jsonl"
TEST = "test.jsonl"

In [ ]:
print("Loading label map and datasets...")
with open("label_map.json", "r") as f:
    label_data = json.load(f)
    label2id = label_data["label2id"]
    id2label = {int(k): v for k, v in label_data["id2label"].items()}

def load_jsonl(filepath):
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_data = load_jsonl("train.jsonl")
val_data = load_jsonl("val.jsonl")

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data)
})


Loading label map and datasets...


In [ ]:
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# We have 13 labels for Volcano NER
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You passed `num_labels=13` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (-100) and convert IDs back to string labels
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [ ]:
use_fp16 = torch.cuda.is_available()

args = TrainingArguments(
    output_dir="./volcano-ner-checkpoints",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,             # 3 is usually enough for this dataset
    weight_decay=0.01,
    eval_strategy="epoch",          # Fixed for newer transformers versions
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    warmup_steps=50,
    logging_steps=20,
    dataloader_num_workers=1,
    fp16=use_fp16,
    report_to="none"
)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer  # Fixed for newer transformers versions
)

In [ ]:
print("\n Starting Volcano NER Fine-Tuning...")
trainer.train()


 Starting Volcano NER Fine-Tuning...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.008208,0.005908,0.980658,0.976879,0.978764
2,0.010192,0.005650,0.986460,0.982659,0.984556
3,0.002115,0.005770,0.993164,0.979769,0.986421
4,0.000736,0.005906,0.990338,0.987476,0.988905
5,0.000009,0.005739,0.992241,0.985549,0.988884


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17220, training_loss=0.006761872686308002, metrics={'train_runtime': 1669.9607, 'train_samples_per_second': 164.977, 'train_steps_per_second': 10.312, 'total_flos': 1.033582997704902e+16, 'train_loss': 0.006761872686308002, 'epoch': 5.0})

In [ ]:
print("\n Saving the best model...")
trainer.save_model("indobert-volcano-ner")
tokenizer.save_pretrained("indobert-volcano-ner")
print(" Training Complete! Model saved to './indobert-volcano-ner'")

print("\nGenerating Validation Classification Report...")
val_results = trainer.predict(dataset["validation"])
val_preds = np.argmax(val_results.predictions, axis=-1)

true_preds = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(val_preds, val_results.label_ids)
]
true_labels = [
    [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(val_preds, val_results.label_ids)
]

print(classification_report(true_labels, true_preds, digits=4))


 Saving the best model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Training Complete! Model saved to './indobert-volcano-ner'

Generating Validation Classification Report...


              precision    recall  f1-score   support

         GOV     0.9659    0.9551    0.9605        89
         HAZ     0.8750    0.9333    0.9032        30
         LOC     0.9963    0.9854    0.9908       274
         SOC     1.0000    0.9963    0.9981       270
        TIME     0.9583    0.9583    0.9583        24
         VOL     0.9972    0.9972    0.9972       351

   micro avg     0.9903    0.9875    0.9889      1038
   macro avg     0.9655    0.9709    0.9680      1038
weighted avg     0.9906    0.9875    0.9890      1038



In [ ]:
import shutil
from google.colab import files

# Replace 'indobert-volcano-ner' with your actual folder name
folder_name = "indobert-volcano-ner"
zip_name = "indobert-volcano-ner.zip"

# Zip the folder
shutil.make_archive(folder_name, 'zip', folder_name)

# Download the zip file
files.download(zip_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>